Validate the Labs Model on eICU data (Patients, Labs, Diagnoses tables)

In [12]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime
import matplotlib.pyplot as plt

In [13]:
#Load eICU tables in from folder eICU
eICU_diagnoses = pd.read_csv(r"eICU\diagnoses.csv")
eICU_lab = pd.read_csv(r"eICU\lab.csv")
eICU_patients = pd.read_csv(r"eICU\patients.csv")



In [14]:
"""
eICU feasibility check for the "drop same-year admissions" fix.

Question: if we can only order stays across years (2014 vs 2015), how many
patients -- and how many BE cases -- actually have an orderable prior stay?

Needs in memory: eICU_patients, eICU_diagnoses
"""

import pandas as pd

# ----------------------------------------------------------------------
# 1. hospital-stay level table (dedupe the unit stays)
# ----------------------------------------------------------------------
hosp = (eICU_patients[['uniquepid', 'patienthealthsystemstayid',
                       'hospitaldischargeyear']]
        .drop_duplicates('patienthealthsystemstayid'))

print(f"unit stays          : {len(eICU_patients):,}")
print(f"hospital stays      : {len(hosp):,}")
print(f"unique patients     : {hosp.uniquepid.nunique():,}")
print(f"discharge years     : {sorted(hosp.hospitaldischargeyear.unique())}")
print()

# ----------------------------------------------------------------------
# 2. how many patients have stays in more than one year?
# ----------------------------------------------------------------------
yr = hosp.groupby('uniquepid')['hospitaldischargeyear'].agg(['nunique', 'min', 'max'])
n_pat = len(yr)
span = yr[yr['nunique'] > 1]

n_stays = hosp.groupby('uniquepid').size()
print(f"patients w/ >1 hospital stay : {(n_stays > 1).sum():,} "
      f"({(n_stays > 1).mean():.2%})")
print(f"patients spanning both years : {len(span):,} ({len(span)/n_pat:.2%})")
print("   ^ this is the ceiling on any orderable-history cohort")
print()

# ----------------------------------------------------------------------
# 3. BE cases
# ----------------------------------------------------------------------
BE_PREFIXES = ('4210', '4211', '4219', 'I33')


def norm(tok):
    return ''.join(ch for ch in str(tok).upper() if ch.isalnum())


dx = eICU_diagnoses[['patientunitstayid', 'icd9code', 'diagnosisstring']].copy()

code_hit = dx['icd9code'].fillna('').apply(
    lambda s: any(norm(t).startswith(BE_PREFIXES) for t in s.split(',') if t.strip())
)
text_hit = dx['diagnosisstring'].str.contains('endocard', case=False, na=False)

be_units = set(dx.loc[code_hit | text_hit, 'patientunitstayid'])

u2p = eICU_patients.set_index('patientunitstayid')[
    ['uniquepid', 'patienthealthsystemstayid', 'hospitaldischargeyear']]
be_rows = u2p.loc[sorted(be_units & set(u2p.index))]

be_pids = set(be_rows.uniquepid)
print(f"BE unit stays  : {len(be_units):,}")
print(f"BE patients    : {len(be_pids):,} "
      f"(crude prevalence {len(be_pids)/n_pat:.3%})")
print(f"   code-only {int((code_hit & ~text_hit).sum()):,} | "
      f"text-only {int((text_hit & ~code_hit).sum()):,} | "
      f"both {int((code_hit & text_hit).sum()):,}")
print()

# ----------------------------------------------------------------------
# 4. THE NUMBER: cases with an orderable prior stay
#    index = earliest BE stay year; need >=1 stay in a strictly earlier year
# ----------------------------------------------------------------------
index_year = be_rows.groupby('uniquepid')['hospitaldischargeyear'].min()

prior = (hosp[hosp.uniquepid.isin(be_pids)]
         .merge(index_year.rename('index_year'), on='uniquepid'))
prior = prior[prior.hospitaldischargeyear < prior.index_year]

usable_cases = set(prior.uniquepid)

print("=" * 62)
print("AFTER DROPPING SAME-YEAR ADMISSIONS")
print("=" * 62)
print(f"  cases spanning both years        : {len(be_pids & set(span.index)):,}")
print(f"  cases w/ a strictly earlier stay : {len(usable_cases):,} "
      f"of {len(be_pids):,}  ({len(usable_cases)/max(len(be_pids),1):.2%})")

ctrl_pids = set(hosp.uniquepid) - be_pids
ctrl_multi = hosp[hosp.uniquepid.isin(ctrl_pids)].groupby('uniquepid')[
    'hospitaldischargeyear'].nunique()
usable_ctrls = int((ctrl_multi > 1).sum())
print(f"  controls spanning both years     : {usable_ctrls:,}")

tot = len(usable_cases) + usable_ctrls
if tot:
    print(f"  resulting cohort                 : {tot:,} patients, "
          f"prevalence {len(usable_cases)/tot:.3%}")
print()
print("  For reference, the current eICU validation run scores")
print("  38,329 patients / 112 cases.")

# how many prior stays does a usable case actually get?
if len(usable_cases):
    depth = prior.groupby('uniquepid').size()
    print(f"\n  prior stays per usable case: "
          f"median {depth.median():.0f}, max {depth.max():.0f}")
    print(f"  {depth.value_counts().sort_index().to_dict()}")

unit stays          : 200,859
hospital stays      : 166,355
unique patients     : 139,367
discharge years     : [np.int64(2014), np.int64(2015)]

patients w/ >1 hospital stay : 18,881 (13.55%)
patients spanning both years : 5,976 (4.29%)
   ^ this is the ceiling on any orderable-history cohort

BE unit stays  : 630
BE patients    : 450 (crude prevalence 0.323%)
   code-only 201 | text-only 0 | both 3,379

AFTER DROPPING SAME-YEAR ADMISSIONS
  cases spanning both years        : 39
  cases w/ a strictly earlier stay : 24 of 450  (5.33%)
  controls spanning both years     : 5,937
  resulting cohort                 : 5,961 patients, prevalence 0.403%

  For reference, the current eICU validation run scores
  38,329 patients / 112 cases.

  prior stays per usable case: median 1, max 4
  {1: 16, 2: 4, 3: 2, 4: 2}


In [15]:
print(f"Diagnoses: {eICU_diagnoses.columns}")
print(eICU_diagnoses.head)
print(f"Labs: {eICU_lab.columns}")
print(eICU_lab.head)
print(f"Patients: {eICU_patients.columns}")
print(eICU_patients.head)

Diagnoses: Index(['diagnosisid', 'patientunitstayid', 'activeupondischarge',
       'diagnosisoffset', 'diagnosisstring', 'icd9code', 'diagnosispriority'],
      dtype='object')
<bound method NDFrame.head of          diagnosisid  patientunitstayid  activeupondischarge  diagnosisoffset  \
0            4222318             141168                False               72   
1            3370568             141168                 True              118   
2            4160941             141168                False               72   
3            4103261             141168                 True              118   
4            3545241             141168                 True              118   
...              ...                ...                  ...              ...   
2710667     46330138            3353251                False            11304   
2710668     46150971            3353251                False             4080   
2710669     46259796            3353254                 True   